# Ngày 9 — RAG cơ bản (OpenAI)

Pipeline: **chunk → embed → cosine search → nhồi ngữ cảnh → trả lời có trích nguồn**.

`pip install openai numpy` · cần `OPENAI_API_KEY` trong env (xem Ngày 6).

In [ ]:
import os
import numpy as np
from openai import OpenAI

assert os.environ.get("OPENAI_API_KEY"), "Chưa đặt OPENAI_API_KEY trong env!"
client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"

## Bộ tài liệu — Sổ tay chính sách IT nội bộ

In [ ]:
DOCS = [
    ("password", "Chính sách mật khẩu: mật khẩu tối thiểu 12 ký tự, phải đổi mỗi 90 ngày, và không được dùng lại 5 mật khẩu gần nhất."),
    ("vpn", "Làm việc từ xa: nhân viên phải kết nối VPN của công ty và bật xác thực hai lớp (2FA) mới truy cập được hệ thống nội bộ."),
    ("email", "Hộp thư: dung lượng email mỗi nhân viên là 50GB. Nếu vượt quá, gửi yêu cầu cho IT để được nâng cấp."),
    ("support", "Hỗ trợ IT: bộ phận IT trực từ 8h00 đến 17h30 các ngày trong tuần. Ngoài giờ, gửi ticket qua hệ thống nội bộ."),
    ("onboarding", "Onboarding: tài khoản cho nhân viên mới được cấp trong vòng 1 ngày làm việc kể từ khi HR gửi yêu cầu."),
    ("device", "Thiết bị: laptop công ty được bảo hành 3 năm. Khi hỏng, báo IT để được cấp máy tạm trong thời gian sửa chữa."),
    ("backup", "Sao lưu: dữ liệu trên OneDrive được sao lưu tự động; dữ liệu lưu trên ổ đĩa cục bộ (ổ D:) KHÔNG được sao lưu."),
    ("offboarding", "Nghỉ việc: tài khoản bị khóa (block credential) ngay trong ngày làm việc cuối cùng; dữ liệu được giữ 90 ngày trước khi xóa."),
]

## 1. Chunk (mỗi đoạn ngắn → 1 chunk; đoạn dài thì cắt theo ký tự)

In [ ]:
def chunk_text(text, size=500):
    return [text[i:i + size] for i in range(0, len(text), size)] or [text]

chunks = []
for source_id, text in DOCS:
    for j, part in enumerate(chunk_text(text)):
        chunks.append({"source_id": source_id, "chunk_id": f"{source_id}#{j}", "text": part})

print(len(chunks), "chunks")

## 2. Embed toàn bộ chunk

In [ ]:
def embed(texts):
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    vecs = np.array([d.embedding for d in resp.data])
    vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)  # chuẩn hóa để cosine = dot
    return vecs

chunk_vecs = embed([c["text"] for c in chunks])
print("Ma trận embedding:", chunk_vecs.shape)

## 3. Retrieve — cosine similarity, lấy top-k

In [ ]:
def retrieve(query, k=3):
    qv = embed([query])[0]
    scores = chunk_vecs @ qv           # cosine vì đã chuẩn hóa
    top = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in top]

for c, s in retrieve("mật khẩu dài bao nhiêu?"):
    print(f"{s:.3f}  [{c['chunk_id']}]  {c['text'][:60]}...")

## 4 + 5. `answer(query)` — trả lời chỉ dựa trên ngữ cảnh, có trích nguồn

In [ ]:
SYSTEM = (
    "Bạn là trợ lý trả lời CHỈ dựa trên NGỮ CẢNH được cung cấp. "
    "Luôn trích nguồn dạng [source_id]. "
    "Nếu ngữ cảnh không chứa thông tin, trả lời đúng câu: 'Không tìm thấy trong tài liệu.' — TUYỆT ĐỐI không bịa."
)

def answer(query, k=3):
    hits = retrieve(query, k)
    context = "\n".join(f"[{c['source_id']}] {c['text']}" for c, _ in hits)
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"NGỮ CẢNH:\n{context}\n\nCÂU HỎI: {query}"},
        ],
        temperature=0,
    )
    sources = [c["source_id"] for c, _ in hits]
    return resp.choices[0].message.content, sources

## Demo

In [ ]:
for q in [
    "Mật khẩu tối thiểu bao nhiêu ký tự?",
    "Làm việc từ xa cần những gì?",
    "Công ty có căng tin miễn phí không?",  # ngoài phạm vi -> phải trả 'không tìm thấy'
]:
    ans, src = answer(q)
    print(f"Q: {q}\nA: {ans}\nNguồn: {src}\n")

## Nhận xét

_(Chunk kích thước nào hợp lý? top-k bao nhiêu là đủ? câu ngoài phạm vi có bị bịa không?)_